In [6]:
import os
import cv2
import numpy as np

In [ ]:
# Paths to ADE20K dataset
IMAGE_DIR = "../images/training"
ANNOTATION_DIR = "../annotations/training"
OUTPUT_LABELS_DIR = "../labels"

In [10]:
# Ensure output directory exists
os.makedirs(OUTPUT_LABELS_DIR, exist_ok=True)

In [ ]:

def convert_to_yolo_format(image_size, bbox):
    """Convert bounding box to YOLO format (x_center, y_center, width, height normalized)"""
    img_w, img_h = image_size
    x_min, y_min, x_max, y_max = bbox
    x_center = (x_min + x_max) / 2.0 / img_w
    y_center = (y_min + y_max) / 2.0 / img_h
    width = (x_max - x_min) / img_w
    height = (y_max - y_min) / img_h
    return x_center, y_center, width, height

In [ ]:

def process_annotations():
    for file in os.listdir(ANNOTATION_DIR):
        if file.endswith(".png"):  # Annotation masks
            mask_path = os.path.join(ANNOTATION_DIR, file)
            img_path = os.path.join(IMAGE_DIR, file.replace(".png", ".jpg"))
            
            # Read image & mask
            image = cv2.imread(img_path)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            height, width = image.shape[:2]
            
            unique_labels = np.unique(mask)
            label_file = os.path.join(OUTPUT_LABELS_DIR, file.replace(".png", ".txt"))

            with open(label_file, "w") as f:
                for label in unique_labels:
                    if label == 0:  # Ignore background
                        continue
                    
                    # Create binary mask for object
                    obj_mask = (mask == label).astype(np.uint8) * 255
                    contours, _ = cv2.findContours(obj_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    
                    for contour in contours:
                        x, y, w, h = cv2.boundingRect(contour)
                        yolo_bbox = convert_to_yolo_format((width, height), (x, y, x + w, y + h))
                        f.write(f"{label} {' '.join(map(str, yolo_bbox))}\n")
    
    print("✅ Conversion completed! YOLO labels saved in", OUTPUT_LABELS_DIR)

In [ ]:
# Run conversion
process_annotations()